# 🚀 SpaceX Falcon 9 Landing Prediction
## Notebook 3 — Data Wrangling

Raw data always needs tidying before analysis. Here we:
- Load the IBM Part 1 dataset
- Audit and handle missing values
- Understand every landing-outcome category
- Encode a binary `Class` label: **1 = successful landing**, **0 = failed / no attempt**


In [19]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', None)


In [20]:
# IBM Part-1 dataset — same source used in notebook 1
df = pd.read_csv(
    "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud"
    "/IBM-DS0321EN-SkillsNetwork/datasets/dataset_part_1.csv"
)
print(f"Shape: {df.shape}")
df.head(10)


Shape: (90, 17)


,FlightNumber,Date,BoosterVersion,PayloadMass,Orbit,LaunchSite,Outcome,Flights,GridFins,Reused,Legs,LandingPad,Block,ReusedCount,Serial,Longitude,Latitude
0,1,2010-06-04,Falcon 9,6104.959412,LEO,CCAFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B0003,-80.577366,28.561857
1,2,2012-05-22,Falcon 9,525.000000,LEO,CCAFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B0005,-80.577366,28.561857
2,3,2013-03-01,Falcon 9,677.000000,ISS,CCAFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B0007,-80.577366,28.561857
3,4,2013-09-29,Falcon 9,500.000000,PO,VAFB SLC 4E,False Ocean,1,False,False,False,NaN,1.0,0,B1003,-120.610829,34.632093
4,5,2013-12-03,Falcon 9,3170.000000,GTO,CCAFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B1004,-80.577366,28.561857
5,6,2014-01-06,Falcon 9,3325.000000,GTO,CCAFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B1005,-80.577366,28.561857
6,7,2014-04-18,Falcon 9,2296.000000,ISS,CCAFS SLC 40,True Ocean,1,False,False,True,NaN,1.0,0,B1006,-80.577366,28.561857
7,8,2014-07-14,Falcon 9,1316.000000,LEO,CCAFS SLC 40,True Ocean,1,False,False,True,NaN,1.0,0,B1007,-80.577366,28.561857
8,9,2014-08-05,Falcon 9,4535.000000,GTO,CCAFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B1008,-80.577366,28.561857
9,10,2014-09-07,Falcon 9,4428.000000,GTO,CCAFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B1011,-80.577366,28.561857


### Missing value audit

In [21]:
null_summary = pd.DataFrame({
    "NullCount":  df.isnull().sum(),
    "NullPct (%)": (df.isnull().sum() / len(df) * 100).round(2)
})
null_summary[null_summary["NullCount"] > 0]


,NullCount,NullPct (%)
LandingPad,26,28.89


### Impute PayloadMass with the column median

In [22]:
median_mass = df['PayloadMass'].median()
df['PayloadMass'].fillna(median_mass, inplace=True)
print(f"PayloadMass NaNs imputed with median = {median_mass:.1f} kg")


PayloadMass NaNs imputed with median = 4701.5 kg


C:\Users\KIIT0001\AppData\Local\Temp\ipykernel_19192\3249492981.py:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['PayloadMass'].fillna(median_mass, inplace=True)


### Understand landing outcome categories

In [23]:
print("Unique Outcome values and counts:")
print(df['Outcome'].value_counts())


Unique Outcome values and counts:
Outcome
True ASDS      41
None None      19
True RTLS      14
False ASDS      6
True Ocean      5
False Ocean     2
None ASDS       2
False RTLS      1
Name: count, dtype: int64


### Define the binary training label — `Class`

| Raw Outcome | Meaning | Class |
|---|---|---|
| `True Ocean` | Controlled ocean landing ✓ | **1** |
| `True RTLS` | Return-to-launch-site landing ✓ | **1** |
| `True ASDS` | Drone-ship landing ✓ | **1** |
| `False Ocean / RTLS / ASDS` | Attempted but failed | **0** |
| `None …` | No landing attempted | **0** |


In [24]:
SUCCESS_OUTCOMES = {"True Ocean", "True RTLS", "True ASDS"}

df['Class'] = df['Outcome'].apply(
    lambda x: 1 if x in SUCCESS_OUTCOMES else 0
)

print("Class distribution:")
print(df['Class'].value_counts())
print(f"Overall landing success rate: {df['Class'].mean():.2%}")


Class distribution:
Class
1    60
0    30
Name: count, dtype: int64
Overall landing success rate: 66.67%


### Success rate per launch site

In [25]:
site_success = (
    df.groupby('LaunchSite')['Class']
      .agg(['mean', 'count'])
      .rename(columns={'mean': 'SuccessRate', 'count': 'Launches'})
      .sort_values('SuccessRate', ascending=False)
)
site_success['SuccessRate'] = site_success['SuccessRate'].map("{:.2%}".format)
print(site_success)


             SuccessRate  Launches
LaunchSite                        
KSC LC 39A        77.27%        22
VAFB SLC 4E       76.92%        13
CCAFS SLC 40      60.00%        55


### Success rate per orbit type

In [26]:
orbit_success = (
    df.groupby('Orbit')['Class']
      .agg(['mean', 'count'])
      .rename(columns={'mean': 'SuccessRate', 'count': 'Launches'})
      .sort_values('SuccessRate', ascending=False)
)
orbit_success['SuccessRate'] = orbit_success['SuccessRate'].map("{:.2%}".format)
print(orbit_success)


      SuccessRate  Launches
Orbit                      
ES-L1     100.00%         1
GEO       100.00%         1
HEO       100.00%         1
SSO       100.00%         5
VLEO       85.71%        14
LEO        71.43%         7
PO         66.67%         9
MEO        66.67%         3
ISS        61.90%        21
GTO        51.85%        27
SO          0.00%         1


### Verify dtypes and final shape

In [27]:
print(df.dtypes)
print(f"Final shape: {df.shape}")
df.tail(5)


FlightNumber        int64
Date               object
BoosterVersion     object
PayloadMass       float64
Orbit              object
LaunchSite         object
Outcome            object
Flights             int64
GridFins             bool
Reused               bool
Legs                 bool
LandingPad         object
Block             float64
ReusedCount         int64
Serial             object
Longitude         float64
Latitude          float64
Class               int64
dtype: object
Final shape: (90, 18)


,FlightNumber,Date,BoosterVersion,PayloadMass,Orbit,LaunchSite,Outcome,Flights,GridFins,Reused,Legs,LandingPad,Block,ReusedCount,Serial,Longitude,Latitude,Class
85,86,2020-09-03,Falcon 9,15400.0,VLEO,KSC LC 39A,True ASDS,2,True,True,True,5e9e3032383ecb6bb234e7ca,5.0,2,B1060,-80.603956,28.608058,1
86,87,2020-10-06,Falcon 9,15400.0,VLEO,KSC LC 39A,True ASDS,3,True,True,True,5e9e3032383ecb6bb234e7ca,5.0,2,B1058,-80.603956,28.608058,1
87,88,2020-10-18,Falcon 9,15400.0,VLEO,KSC LC 39A,True ASDS,6,True,True,True,5e9e3032383ecb6bb234e7ca,5.0,5,B1051,-80.603956,28.608058,1
88,89,2020-10-24,Falcon 9,15400.0,VLEO,CCAFS SLC 40,True ASDS,3,True,True,True,5e9e3033383ecbb9e534e7cc,5.0,2,B1060,-80.577366,28.561857,1
89,90,2020-11-05,Falcon 9,3681.0,MEO,CCAFS SLC 40,True ASDS,1,True,False,True,5e9e3032383ecb6bb234e7ca,5.0,0,B1062,-80.577366,28.561857,1


### Save wrangled dataset

In [28]:
df.to_csv("../data/spacex_wrangled.csv", index=False)
print("Saved → data/spacex_wrangled.csv")


Saved → data/spacex_wrangled.csv
